In [1]:
library(tidyverse)
library(pheatmap)
library(data.table)

reverse_complement <- function(dna_seq) {
  complement <- c("A" = "T", "T" = "A", "C" = "G", "G" = "C")
  nucleotides <- unlist(strsplit(dna_seq, ""))
  complement_nucleotides <- complement[nucleotides]
  reverse_complement_seq <- paste(rev(complement_nucleotides), collapse = "")
  return(reverse_complement_seq)
}

twist_barcodes <- read_csv("/mnt/dawnccle2/melange/data/guide_library/20230130_twist_library_v3.csv") %>%
  mutate(barcodeRevcomp = sapply(barcode, reverse_complement))

######## Look at rmats stats ########
combined_psi <- read_tsv("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_altSS/MUT_altSS_combined_output_indiv.tsv")
calculate_ratio <- function(I, S) {
  I_values <- as.numeric(unlist(strsplit(I, ",")))
  S_values <- as.numeric(unlist(strsplit(S, ",")))
  ratio <- I_values / (I_values + S_values)
  return(paste(round(ratio,3), collapse = ","))
}

calculate_average <- function(PSI){
  PSI_values <- as.numeric(unlist(strsplit(PSI, ",")))
  average <- mean(PSI_values)
  return(round(average, 3))
}

calculate_average_count_sum <- function(I, S){
  I_values <- as.numeric(unlist(strsplit(I, ",")))
  S_values <- as.numeric(unlist(strsplit(S, ",")))
  total_sum <- I_values + S_values
  average_count_sum <- mean(total_sum)
  return(round(average_count_sum, 0))
}

# Apply the function to the data frame
combined_psi <- combined_psi %>%
  mutate(
    PSI1 = mapply(calculate_ratio, I1, S1),
    PSI2 = mapply(calculate_ratio, I2, S2)
  ) %>% 
  mutate(
    PSI1_average = mapply(calculate_average, PSI1),
    PSI2_average = mapply(calculate_average, PSI2)
  ) %>%
  mutate(PSI_diff = PSI2_average - PSI1_average) %>% 
  mutate(
    count_sum_average1 = mapply(calculate_average_count_sum, I1, S1),
    count_sum_average2 = mapply(calculate_average_count_sum, I2, S2)
  ) %>% mutate(PSI_ratio = PSI2_average / PSI1_average) %>% 
  mutate(PSI_reverse_ratio = (1-PSI2_average)/(1-PSI1_average))

wanted_pairs <- c(
#'MUT2_CH3-1_A1_CH3-1_A2',
'MUT_CH3-1_A1_FUBP1_B12',
'MUT_CH3-1_A1_FUBP1_C5',
'MUT_CH3-1_A1_RBM10_C8',
'MUT_CH3-1_A1_RBM10_G4',
'MUT_CH3-1_A1_RBM5_A2',
'MUT_CH3-1_A1_RBM5_A3',
'MUT_CH3-1_A1_ZRSR2_F8',
'MUT_CH3-1_A1_ZRSR2_G9',
'MUT_CH3-1_A2_FUBP1_B12',
'MUT_CH3-1_A2_FUBP1_C5',
'MUT_CH3-1_A2_RBM10_C8',
'MUT_CH3-1_A2_RBM10_G4',
'MUT_CH3-1_A2_RBM5_A2',
'MUT_CH3-1_A2_RBM5_A3',
'MUT_CH3-1_A2_ZRSR2_F8',
'MUT_CH3-1_A2_ZRSR2_G9',
'MUT_K562WT_K562K700E',
'MUT_MUT-plx317_U2AF1_WT_MUT-plx317_U2AF1_Q157A',
'MUT_MUT-plx317_U2AF1_WT_MUT-plx-LacZ',
'MUT_MUT-sgCh3-1_MUT-sgRBM10',
'MUT_MUT-sgCh3-1_MUT-sgRBM5',
'MUT_MUT-sgCh3-1_MUT-sgRUBP1',
'MUT_splicelib_sgCh3_splicelib_ZRSR2',
'MUT_splicelib_U2AF1_WT_splicelib_U2AF1_S34F')

same_pairs <- c(
'MUT_CH3-1_A1_CH3-1_A2',
'MUT_FUBP1_B12_FUBP1_C5',
'MUT_RBM10_C8_RBM10_G4',
'MUT_RBM5_A2_RBM5_A3',
'MUT_ZRSR2_F8_ZRSR2_G9')



── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘data.table’


The following objects are masked from ‘package:lubridate’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


The following objects are masked from ‘package:dplyr’:

    between, first, last


The following object is masked from ‘package:purrr’:

    transpose


Rows: 46372 Columns: 7
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (7): ID, barcode, u

In [2]:

# Get sequences in folders MUT2_CH3-1_A1
all_ch3_a1 <- combined_psi %>% filter(grepl("MUT_CH3-1_A1", Folder)) 
chr3_a1_seq_count <- all_ch3_a1 %>% 
group_by(ExonID) %>% 
summarise(count = n()) %>% 
arrange(desc(count))

seq_to_exclude_a1 <- chr3_a1_seq_count %>% 
filter(count >=4) %>% 
pull(ExonID)

# Also exclude sequences in MUT2_CH3-1_A2
all_ch3_a2 <- combined_psi %>% filter(grepl("MUT_CH3-1_A2", Folder)) 
chr3_a2_seq_count <- all_ch3_a2 %>% 
group_by(ExonID) %>% 
summarise(count = n()) %>% 
arrange(desc(count))

seq_to_exclude_a2 <- chr3_a2_seq_count %>% 
filter(count >=4) %>% 
pull(ExonID)

# Also look at diff between A1 and A2
control_de <- combined_psi %>% 
filter(Folder == "MUT2_CH3-1_A1_CH3-1_A2") %>% 
filter(count_sum_average1 >30) %>% 
filter(count_sum_average2 > 30) %>% 
mutate(log2_PSI_ratio = log2(PSI_ratio), log2_PSI_reverse_ratio = log2(PSI_reverse_ratio)) %>% 
  separate(ExonID, sep = "__", into =c("index", "offset"), remove = FALSE) %>%
  separate(offset, into = c("skipped_exon_start", "skipped_exon_end", "downstream_exon_start"), sep = ":", remove = FALSE) %>%
  filter(abs(as.integer(skipped_exon_start)) != 1 & abs(as.integer(skipped_exon_end)) != 1) 

seq_to_exclude_control <- control_de %>% 
pull(ExonID) %>% 
unique()

all_seq_to_exclude <- c(seq_to_exclude_a1, seq_to_exclude_a2, seq_to_exclude_control)

combined_psi_filtered <- combined_psi %>% 
  filter(!ExonID %in% all_seq_to_exclude) %>% 
  # filter(Folder %in% wanted_pairs) %>%
  filter(count_sum_average1 >30) %>% 
  filter(count_sum_average2 > 30) %>% 
  mutate(log2_PSI_ratio = log2(PSI_ratio), log2_PSI_reverse_ratio = log2(PSI_reverse_ratio)) %>% 
  separate(ExonID, sep = "__", into =c("index", "offset"), remove = FALSE) %>%
  separate(offset, into = c("skipped_exon_start", "skipped_exon_end", "downstream_exon_start"), sep = ":", remove = FALSE) %>%
  filter(abs(as.integer(skipped_exon_start)) != 1 & abs(as.integer(skipped_exon_end)) != 1) 

######## Process and Save the Individual Sequences as Fasta ######
# Define a function to process and save sequences
process_and_save <- function(df, name) {
  # Define file paths
  upstream_fasta <- paste0("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_altSS/seq_output/", name, "_upstreamIntronSeq_adj.fasta")
  skipped_fasta <- paste0("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_altSS/seq_output/", name, "_skippedExonSeq_adj.fasta")
  downstream_fasta <- paste0("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_altSS/seq_output/", name, "_downstreamIntronSeq_adj.fasta")
  
  # Write upstream intron sequences
  fileConn <- file(upstream_fasta, "w")
  apply(df, 1, function(row) {
    cat(paste0(">", paste0(row["barcode"], "_", row["offset"]), "\n", row["upstreamIntronSeq_adj"], "\n"), file = fileConn)
  })
  close(fileConn)
  
  # Write skipped exon sequences
  fileConn <- file(skipped_fasta, "w")
  apply(df, 1, function(row) {
    cat(paste0(">", paste0(row["barcode"], "_", row["offset"]), "\n", row["skippedExonSeq_adj"], "\n"), file = fileConn)
  })
  close(fileConn)
  
  # Write downstream intron sequences
  fileConn <- file(downstream_fasta, "w")
  apply(df, 1, function(row) {
    cat(paste0(">", paste0(row["barcode"], "_", row["offset"]), "\n", row["downstreamIntronSeq_adj"], "\n"), file = fileConn)
  })
  close(fileConn)
  
  # Save CSV file
  csv_path <- paste0("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_altSS/seq_output/MUT_altSS_", name, "_seq.csv")
  write_csv(df, csv_path)
  
  # Message to indicate completion
  cat("FASTA and CSV files saved for:", name, "\n")
}

# Process and save each dataset
unique_folders <- unique(combined_psi_filtered$Folder)

# Create a list of datasets for all unique folders
datasets <- list()
for (folder in unique_folders) {
   # folder_name <- gsub("MUT2_", "", folder)  # Remove the MUT2_ prefix if present
  datasets[[folder]] <- combined_psi_filtered %>% filter(Folder == folder)
}

# Print the number of datasets to be processed
cat("Processing", length(datasets), "unique datasets\n")

for (name in names(datasets)) {
  df <- datasets[[name]] %>%
    left_join(twist_barcodes, by = c("index" = "ID")) %>%
    mutate(
      upstream_offset = as.integer(skipped_exon_start),
      downstream_offset = as.integer(skipped_exon_end),
      const_offset = as.integer(downstream_exon_start),
      upstreamIntron_len = nchar(upstreamIntronSeq),
      downstreamIntron_len = nchar(downstreamIntronSeq),
      skippedExon_len = nchar(skippedExonSeq),
      upstreamIntron_len_adj = upstreamIntron_len + upstream_offset,
      downstreamIntron_len_adj = downstreamIntron_len - downstream_offset,
      skippedExon_len_adj = skippedExon_len - upstream_offset + downstream_offset,
      upstreamIntronSeq_adj = substr(librarySequence, 1, upstreamIntron_len_adj),
      skippedExonSeq_adj = substr(librarySequence, upstreamIntron_len_adj + 1, upstreamIntron_len_adj + skippedExon_len_adj),
      downstreamIntronSeq_adj = substr(librarySequence, upstreamIntron_len_adj + skippedExon_len_adj + 1, upstreamIntron_len_adj + skippedExon_len_adj + downstreamIntron_len_adj)
    )
  
  process_and_save(df, name)
}

cat("Processing complete for all datasets.\n")

# Make the "background sequence" file 
upstream_fasta <- paste0("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_altSS/seq_output/", "background_upstreamIntronSeq_adj.fasta")
skipped_fasta <- paste0("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_altSS/seq_output/", "background_skippedExonSeq_adj.fasta")
downstream_fasta <- paste0("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_altSS/seq_output/", "background_downstreamIntronSeq_adj.fasta")

# Write upstream intron sequences
fileConn <- file(upstream_fasta, "w")
apply(twist_barcodes, 1, function(row) {
  cat(paste0(">", paste0(row["barcode"], "_", row["offset"]), "\n", row["upstreamIntronSeq"], "\n"), file = fileConn)
})
close(fileConn)

# Write skipped exon sequences
fileConn <- file(skipped_fasta, "w")
apply(twist_barcodes, 1, function(row) {
  cat(paste0(">", paste0(row["barcode"], "_", row["offset"]), "\n", row["skippedExonSeq"], "\n"), file = fileConn)
})
close(fileConn)

# Write downstream intron sequences
fileConn <- file(downstream_fasta, "w")
apply(twist_barcodes, 1, function(row) {
  cat(paste0(">", paste0(row["barcode"], "_", row["offset"]), "\n", row["downstreamIntronSeq"], "\n"), file = fileConn)
})
close(fileConn)

Warning message:
“Expected 2 pieces. Additional pieces discarded in 64 rows [1, 2, 3, 4, 5, 6, 7,
8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, ...].”


Processing 14 unique datasets
FASTA and CSV files saved for: MUT_CH3-1_A1_FUBP1_B12 
FASTA and CSV files saved for: MUT_CH3-1_A1_RBM10_C8 
FASTA and CSV files saved for: MUT_CH3-1_A1_RBM5_A2 
FASTA and CSV files saved for: MUT_CH3-1_A1_RBM5_A3 
FASTA and CSV files saved for: MUT_CH3-1_A1_ZRSR2_F8 
FASTA and CSV files saved for: MUT_CH3-1_A1_ZRSR2_G9 
FASTA and CSV files saved for: MUT_CH3-1_A2_FUBP1_B12 
FASTA and CSV files saved for: MUT_CH3-1_A2_RBM10_C8 
FASTA and CSV files saved for: MUT_CH3-1_A2_RBM10_G4 
FASTA and CSV files saved for: MUT_CH3-1_A2_RBM5_A2 
FASTA and CSV files saved for: MUT_CH3-1_A2_RBM5_A3 
FASTA and CSV files saved for: MUT_K562WT_K562K700E 
FASTA and CSV files saved for: MUT_splicelib_sgCh3_splicelib_ZRSR2 
FASTA and CSV files saved for: MUT_splicelib_U2AF1_WT_splicelib_U2AF1_S34F 
Processing complete for all datasets.


NULL

NULL

NULL

In [3]:
# Plot the volcano plot
# save the plot
p1 <- ggplot(combined_psi_filtered, aes(log2(PSI_ratio), -log10(FDR))) + 
  geom_point() + 
  facet_wrap(~Folder, nrow = 4) +
  theme_classic() +
  ggtitle("PSI ratio")
ggsave("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_altSS/seq_output/MUT_volcano_plot_altSS_ratio.png", p1, width = 20, height = 20)

# Plot PSI_reverse_ratio
p2 <- ggplot(combined_psi_filtered, aes(log2(PSI_reverse_ratio), -log10(FDR))) + 
  geom_point() + 
  facet_wrap(~Folder, nrow = 4) +
  theme_classic() +
  ggtitle("PSI reverse ratio")
ggsave("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_altSS/seq_output/MUT_volcano_plot_altSS_reverse_ratio.png", p2, width = 20, height = 20)

# Also plot the PSI_dff
p3 <- ggplot(combined_psi_filtered, aes(PSI_diff, -log10(FDR))) + 
  geom_point() + 
  facet_wrap(~Folder, nrow = 4) +
  theme_classic() +
  ggtitle("PSI diff")
ggsave("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_altSS/seq_output/MUT_volcano_plot_altSS_diff.png", p3, width = 20, height = 20)
 


In [4]:

# Across all samples, get log2(PSI_ratio) > 2
high_ratio <- combined_psi_filtered %>% 
filter(log2_PSI_ratio >= 2 | log2_PSI_reverse_ratio >= 2) %>%
filter(abs(PSI_diff) >= 0.2) %>% 
arrange(desc(pmax(log2_PSI_ratio, log2_PSI_reverse_ratio))) %>%
    left_join(twist_barcodes, by = c("index" = "ID")) %>%
    mutate(
      upstream_offset = as.integer(skipped_exon_start),
      downstream_offset = as.integer(skipped_exon_end),
      const_offset = as.integer(downstream_exon_start),
      upstreamIntron_len = nchar(upstreamIntronSeq),
      downstreamIntron_len = nchar(downstreamIntronSeq),
      skippedExon_len = nchar(skippedExonSeq),
      upstreamIntron_len_adj = upstreamIntron_len + upstream_offset,
      downstreamIntron_len_adj = downstreamIntron_len - downstream_offset,
      skippedExon_len_adj = skippedExon_len - upstream_offset + downstream_offset,
      upstreamIntronSeq_adj = substr(librarySequence, 1, upstreamIntron_len_adj),
      skippedExonSeq_adj = substr(librarySequence, upstreamIntron_len_adj + 1, upstreamIntron_len_adj + skippedExon_len_adj),
      downstreamIntronSeq_adj = substr(librarySequence, upstreamIntron_len_adj + skippedExon_len_adj + 1, upstreamIntron_len_adj + skippedExon_len_adj + downstreamIntron_len_adj)
    )

write_csv(high_ratio, "/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_altSS/seq_output/MUT_high_ratio_altSS.csv")
